<a href="https://colab.research.google.com/github/vaibhav2017goyal/Angular/blob/main/optim_%2B_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import os
import time
import logging
import requests

# ---------------- MOUNT DRIVE ----------------
drive.mount('/content/drive')

# ---------------- LOGGING ----------------
logging.basicConfig(filename='/content/nifty_errors.log', level=logging.ERROR)

# ---------------- DOWNLOAD NSE STOCK LIST ----------------
nse_url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"
csv_file_path = "/content/drive/MyDrive/Nifty50/EQUITY_L.csv"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9"
}

response = requests.get(nse_url, headers=headers)

if response.status_code == 200:
    with open(csv_file_path, "wb") as f:
        f.write(response.content)
    print("✅ NSE stock list updated")
else:
    print("❌ Failed to download NSE list")

# ---------------- READ STOCK SYMBOLS ----------------
df_stocks = pd.read_csv(csv_file_path)
symbols = df_stocks['SYMBOL'].apply(lambda x: x.strip() + '.NS').tolist()

# ---------------- FUNCTIONS ----------------

def get_return(hist, months):
    if hist.empty:
        return np.nan
    end = hist.index[-1]
    start = end - pd.DateOffset(months=months)
    data = hist[hist.index >= start]
    if len(data) == 0:
        return np.nan
    return ((data['Close'].iloc[-1] / data['Close'].iloc[0]) - 1) * 100

def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return round(rsi.iloc[-1], 2)

# ---------------- MAIN DATA FETCH ----------------
def get_stock_data():
    results = []

    for i, symbol in enumerate(symbols):
        try:
            print(f"Processing {i+1}/{len(symbols)}: {symbol}")
            time.sleep(1)  # reduce rate limit risk

            stock = yf.Ticker(symbol)
            hist = stock.history(period='3y')

            if hist.empty:
                continue

            info = stock.fast_info

            # Returns
            r1 = get_return(hist, 1)
            r6 = get_return(hist, 6)
            r12 = get_return(hist, 12)
            r36 = get_return(hist, 36)

            # RSI
            rsi = calculate_rsi(hist['Close'])

            # Data append
            results.append({
                'Name': symbol.replace('.NS', ''),
                'CMP Rs.': round(hist['Close'].iloc[-1], 2),
                'Vol 1d': hist['Volume'].iloc[-1],
                'EPS 36M Rs.': stock.info.get('trailingEps', np.nan),
                'Mar Cap Rs.Cr.': round(info.get('marketCap', np.nan) / 1e7, 2) if info.get('marketCap') else np.nan,
                '1mth return %': round(r1, 2),
                '6mth return %': round(r6, 2),
                '12mth return %': round(r12, 2),
                '36mth return %': round(r36, 2),
                'RSI': rsi
            })

        except Exception as e:
            print(f"Error {symbol}: {e}")
            logging.error(f"{symbol} error: {e}")

    return pd.DataFrame(results)

# ---------------- FETCH DATA ----------------
print("\nFetching stock data...")
df = get_stock_data()

# ---------------- SORT ----------------
df.sort_values(by=['12mth return %', '36mth return %'], ascending=[False, False], inplace=True)

# ---------------- FILTER LOGIC ----------------

# Convert volume to numeric
df['Vol 1d'] = pd.to_numeric(df['Vol 1d'], errors='coerce')

# Average volume
avg_volume = df['Vol 1d'].mean()
print(f"\nAverage Volume: {avg_volume:,.0f}")

# Apply filters
target_df = df[
    (df['Vol 1d'] > avg_volume) &
    (df['EPS 36M Rs.'] > 0) &
    (df['1mth return %'] < 0) &
    (df['6mth return %'] > 0) &
    (df['Mar Cap Rs.Cr.'] > 5000)
]

# Sort target
target_df = target_df.sort_values(by=['6mth return %', 'RSI'], ascending=[False, True])

print("\nFiltered Target Stocks:")
print(target_df.to_string(index=False))

# ---------------- SAVE TO EXCEL ----------------
today = datetime.today().strftime('%Y-%m-%d')
save_dir = '/content/drive/MyDrive/Nifty50'
os.makedirs(save_dir, exist_ok=True)

file_path = os.path.join(save_dir, f"NSE_Stock_Analysis_{today}.xlsx")

with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='All_Data', index=False)
    target_df.to_excel(writer, sheet_name='Target', index=False)

print(f"\n✅ Data saved to: {file_path}")

Mounted at /content/drive
✅ NSE stock list updated

Fetching stock data...
Processing 1/2593: 20MICRONS.NS
Processing 2/2593: 21STCENMGM.NS
Processing 3/2593: 360ONE.NS
Processing 4/2593: 3BBLACKBIO.NS
Processing 5/2593: 3IINFOLTD.NS
Processing 6/2593: 3MINDIA.NS
Processing 7/2593: 3PLAND.NS
Processing 8/2593: 5PAISA.NS
Processing 9/2593: 63MOONS.NS
Processing 10/2593: A2ZINFRA.NS
Processing 11/2593: AAATECH.NS
Processing 12/2593: AADHARHFC.NS
Processing 13/2593: AAKASH.NS
Processing 14/2593: AAREYDRUGS.NS
Processing 15/2593: AARNAV.NS
Processing 16/2593: AARON.NS
Processing 17/2593: AARTECH.NS
Processing 18/2593: AARTIDRUGS.NS
Processing 19/2593: AARTIIND.NS
Processing 20/2593: AARTIPHARM.NS
Processing 21/2593: AARTISURF.NS
Processing 22/2593: AARVI.NS
Processing 23/2593: AASTHA.NS
Processing 24/2593: AAVAS.NS
Processing 25/2593: ABAN.NS
Processing 26/2593: ABANSENT.NS
Processing 27/2593: ABB.NS
Processing 28/2593: ABBOTINDIA.NS
Processing 29/2593: ABCAPITAL.NS
Processing 30/2593: ABC

ERROR:yfinance:$KALYANI.NS: possibly delisted; no price data found  (period=5d)


Processing 1183/2593: KALYANIFRG.NS
Processing 1184/2593: KALYANKJIL.NS
Processing 1185/2593: KAMAHOLD.NS
Processing 1186/2593: KAMANWALA.NS
Processing 1187/2593: KAMATHOTEL.NS
Processing 1188/2593: KAMDHENU.NS
Processing 1189/2593: KAMOPAINTS.NS
Processing 1190/2593: KANANIIND.NS
Processing 1191/2593: KANCHI.NS
Processing 1192/2593: KANCOTEA.NS
Processing 1193/2593: KANOHAR.NS
Processing 1194/2593: KANORICHEM.NS
Processing 1195/2593: KANPRPLA.NS
Processing 1196/2593: KANSAINER.NS
Processing 1197/2593: KAPSTON.NS
Processing 1198/2593: KARAMTARA.NS
Processing 1199/2593: KARMAENG.NS
Processing 1200/2593: KARURVYSYA.NS
Processing 1201/2593: KAUSHALYA.NS
Processing 1202/2593: KAVDEFENCE.NS
Processing 1203/2593: KAYA.NS
Processing 1204/2593: KAYNES.NS
Processing 1205/2593: KCP.NS
Processing 1206/2593: KCPSUGIND.NS
Processing 1207/2593: KDDL.NS
Processing 1208/2593: KDGREEN.NS
Processing 1209/2593: KEC.NS
Processing 1210/2593: KECL.NS
Processing 1211/2593: KEEPLEARN.NS
Processing 1212/2593: 

ERROR:yfinance:$SAB.NS: possibly delisted; no price data found  (period=5d)


Processing 1945/2593: SADBHAV.NS
Processing 1946/2593: SADBHIN.NS
Processing 1947/2593: SADHNANIQ.NS
Processing 1948/2593: SAFARI.NS
Processing 1949/2593: SAGARDEEP.NS
Processing 1950/2593: SAGCEM.NS
Processing 1951/2593: SAGILITY.NS
Processing 1952/2593: SAHLIBHFI.NS
Processing 1953/2593: SAHYADRI.NS
Processing 1954/2593: SAICAPI.NS
Processing 1955/2593: SAIL.NS
Processing 1956/2593: SAILIFE.NS
Processing 1957/2593: SAIPARENT.NS
Processing 1958/2593: SAKAR.NS
Processing 1959/2593: SAKHTISUG.NS
Processing 1960/2593: SAKSOFT.NS
Processing 1961/2593: SAKUMA.NS
Processing 1962/2593: SALASAR.NS
Processing 1963/2593: SALONA.NS
Processing 1964/2593: SALSTEEL.NS
Processing 1965/2593: SALZERELEC.NS
Processing 1966/2593: SAMBANDAM.NS
Processing 1967/2593: SAMBHAAV.NS
Processing 1968/2593: SAMBHV.NS
Processing 1969/2593: SAMHI.NS
Processing 1970/2593: SAMMAANCAP.NS
Processing 1971/2593: SAMPANN.NS
Processing 1972/2593: SANATHAN.NS
Processing 1973/2593: SANDESH.NS
Processing 1974/2593: SANDHAR.NS